# 4_figures/02 — Generate manuscript figure data

Runs the manuscript preparation modules in `figures/prep/` with the active **Python kernel**. Each reads from
`DATA_PATH` and writes CSVs to `DATA_PATH/figure_data/` (`config.FIGURE_DATA_DIR`, overridable via
the `CTEP_FIGURE_DATA_DIR` env var), which the R rendering tier (`03_render_figures.R`) plots
from. `R/config.R` defines the same location for the R side — if you override it, override it for
both tiers or the plots will read the old directory.

- **Code lookups (R)**: [01_code_lookups.R](01_code_lookups.R) — one-time bootstrap.
- **Prep tier**: this notebook (Python).
- **Render tier**: [03_render_figures.R](03_render_figures.R) (R Markdown).

### Incremental by default

A module is **skipped when all of its output CSVs already exist**, so a re-run regenerates only what
is actually missing. Most of these modules re-read large parquets and refit clustering or scoring
work to produce a handful of small CSVs, so skipping a satisfied module is the difference between
seconds and many minutes.

The skip is deliberately coarse — whole module, not per-CSV — because these modules share
intermediates internally (`figure2`'s `stage_vs_risk_df` feeds three outputs, `figure4`'s clustering
feeds five), so producing one missing CSV costs essentially the same as producing all of them.

**The check is presence, not freshness.** It cannot tell a stale CSV from a current one. After
anything upstream changes — a cohort rebuild, new trajectories from `2_models/03`, updated model
metrics or held-out risk scores, a re-run biomarker pipeline — set `FORCE` for the affected modules,
or `REGENERATE_ALL = True`.

Note that the check is also blind to the directory *moving*: pointing `FIGURE_DATA_DIR` at a new,
empty location makes every module report as missing and re-run in full, which is the intended
behavior on a first run against a fresh target dir.

**Prerequisite:** run `4_figures/01` once before the first run of this notebook, and again after a cohort rebuild. This notebook
does not invoke R — absent lookups make figure2 fall back to raw codes and log the miss counts
rather than failing; the setup cell tells you which case you are in.

**Within-cancer supplements:** `figures.prep.within_cancer` evaluates the existing pooled models
within each recorded cancer type; it does not fit cancer-specific models and does not depend on
`2_models/02_within_vs_pan.ipynb`. It pairs text and base scores from `full_cohort_risk_scores/`
(for example, after `2_models/04`) and text and other-modality scores from `held_out_risk_scores/`
(after the feature-comparison and held-out-risk jobs). C-indices are computed within joint
text/comparator outer-fold blocks and combined by their comparable-pair counts. Each endpoint
and cancer-type comparison requires at least 20 matched patients, 5 events, and 1 comparable pair
by default. Use `EXTRA_ARGS["within_cancer"]` for `--min-patients` / `--min-events` overrides.
This module is independently incremental: use `ONLY = {"within_cancer"}` to prepare just the
supplements or `FORCE = {"within_cancer"}` after score, cohort, or threshold changes. Its audit
CSV records unavailable or ineligible comparisons; rendering uses only eligible (`status = ok`)
rows and applies the shared manuscript endpoint filter.
It also writes `fig3_within_cancer_modality_cindex.csv`: every modality's C-index on one
shared set of patients and comparable pairs per endpoint and cancer type (blocks joint over
all modalities' outer folds), which the within-cancer modality ranks use.

Before evaluating each endpoint, this module calculates valid patient, observed-event,
and non-event (`n_non_events`) counts by cancer type, saved in
`fig2_within_cancer_event_counts.csv` and `fig3_within_cancer_event_counts.csv`,
including endpoints without trained risk scores and strata with zero eligible patients.
Count rows record eligibility, status, and the required patient/event thresholds.
Full-cohort membership is scheme-specific embedding IDs intersected with cancer
annotations; the modality cohort also intersects somatic, germline, stage, and
treatment IDs as in training. Metastatic burden is zero-filled and does not restrict
membership. Counts are upper bounds before matching predictions, so final paired
cohorts must still pass the thresholds; undersized strata skip concordance.
Two global progress bars, **Full cohort** and **Modality cohort**, cover all
schemes/endpoints. Warnings are suppressed; skips and missing modality inputs are
recorded in `within_cancer_audit.csv` without console logs.

Risk-score files require `outer_fold`; legacy files without it are skipped and
audited. Regenerate them with the corresponding training/risk runner's `--overwrite`
option. Even with `ONLY = {"within_cancer"}`, rendering with the shared endpoint
filter requires `fig2_full_cohort_metrics.csv`: run `figure2` preparation once or
set `MANUSCRIPT_FILTER_UNDERPERFORMING_ENDPOINTS=false` to render without that filter.

**Published prognostic scores:** `figures.prep.published_scores` compares published within-cancer-type prognostic scores (MDCalc-style: mGPS, RMH, LIPI, ALBI, MELD, CAPRA-mod, and ECOG-free IPI/IMDC/MSKCC) against the held-out full-cohort text risk score for overall survival. For each score it evaluates three models on the same eligible, lab-observable, complete-case patients and comparable pairs: the published score alone, text alone, and the two combined. It reads `published_scores_df*.csv.gz` written by the standalone `notebooks/1_data/01b_published_scores.ipynb` pipeline (`pipelines.preprocessing.audit_published_score_inputs` then `build_published_scores`) and writes `pubscore_cindex.csv`, `pubscore_delta.csv`, `pubscore_cox.csv`, `pubscore_km.csv`, and `pubscore_cohort.csv`. Only the primary treatment-anchor, 30-day lab-window run is plotted; the sequencing-anchor and 90-day-window runs are sensitivity results kept in the report tables only. Use `FORCE = {"published_scores"}` after held-out text risk scores or published-score inputs change.

**Within-cancer joint Cox:** `figures.prep.within_cancer_joint` refits the Figure 3 joint Cox
model within each selected cancer type (`SELECTED_CANCER_TYPES` in `shared/palette.json`), with
the same inputs, eligibility rules and fit variants as `fig3_joint_betas.csv`, standardizing
scores within the cancer type. It writes `fig3_within_cancer_joint_betas.csv` and a per-stratum
status table, `fig3_within_cancer_joint_fits.csv`. Use `FORCE = {"within_cancer_joint"}` after
held-out risk scores change.

**Combined-modality models:** `figures.prep.figure3_combined` fits stacked Cox models on the
held-out modality risk scores for every endpoint: each non-text modality alone and with text,
text alone, all modalities except text, and all modalities. Scores are standardized within each
modality's outer fold and the stacking fit is cross-fitted over the text model's outer folds;
C-indices share one cohort and one set of comparable pairs per endpoint. It writes
`fig3_combined_cindex.csv` and, for overall survival, bootstrap intervals in
`fig3_combined_os_cindex.csv` and `fig3_combined_os_delta.csv` (`EXTRA_ARGS["figure3_combined"]`
takes `--n-boot`). Use `FORCE = {"figure3_combined"}` after held-out risk scores change.

**Within-cancer KM:** `figures.prep.within_cancer_km` splits the Figure 2c/d cohort (known major
stage, full-cohort overall-survival text risk score) by selected cancer type, keeps the pan-cancer text
risk quartiles of Figure 2d (cut before the split), and writes `fig2_km_stage_vs_risk_by_cancer.csv` and
`fig2_stage_vs_risk_cindex_by_cancer.csv`. Types below 20 patients or 5 deaths are recorded with a
status and not plotted. Use `FORCE = {"within_cancer_km"}` after full-cohort risk scores change.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "figures").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import CODE_PATH, FIGURE_DATA_DIR  # noqa: E402

print(f"repo root:     {REPO_ROOT}")
print(f"figure data: {FIGURE_DATA_DIR}")

# Preflight: figure2's phecode labels come from the lookups 4_figures/01 builds. Absent lookups are a
# degraded run, not a failure — say so loudly rather than silently.
_missing = [name for name in ("icd10_to_phecode_mapping.csv", "phecode_descriptions.csv")
            if not os.path.exists(os.path.join(CODE_PATH, name))]
if _missing:
    print("\nWARNING: missing code lookups: " + ", ".join(_missing)
          + "\n  figure2 will fall back to raw codes and disable cross-scheme event dedup."
          + "\n  Run 01_code_lookups.R for manuscript-quality labels.")
else:
    print("code lookups: present (4_figures/01 has run).")

## Configuration

`OUTPUTS` mirrors the `save_figure_data` calls in each module. If a module gains or loses an output,
update it here too — an entry that overstates a module's outputs makes it always re-run; one that
understates makes it skip while a CSV is still missing.

In [ ]:
# module -> the figure-data CSVs its main() writes
OUTPUTS: dict[str, list[str]] = {
    "figure0": [
        "fig0_data_availability.csv", "fig0_availability_combinations.csv",
    ],
    "figure1": [
        "fig1_endpoint_counts.csv", "fig1_cancer_type_counts.csv", "fig1_stage_counts.csv",
    ],
    "figure2": [
        "fig2_full_cohort_metrics.csv",
        "fig2_km_stage_vs_risk.csv", "fig2_stage_vs_risk_cindex.csv",
        "fig2_stage_vs_risk_cindex_by_stage.csv",
    ],
    "figure2_anchor": [
        "fig2_anchor_sensitivity.csv", "fig2_anchor_cohort_overlap.csv",
    ],
    "figure3": [
        "fig3_modality_cindex.csv",
        "fig3_modality_avg_rank_cindex.csv",
        "fig3_modality_ranks_long_cindex.csv",
        "fig3_joint_betas.csv",
    ],
    "within_cancer": [
        "fig2_within_cancer_cindex.csv", "fig3_within_cancer_cindex.csv",
        "fig2_within_cancer_event_counts.csv", "fig3_within_cancer_event_counts.csv",
        "within_cancer_audit.csv", "fig3_within_cancer_modality_cindex.csv",
    ],
    "within_cancer_joint": [
        "fig3_within_cancer_joint_betas.csv", "fig3_within_cancer_joint_fits.csv",
    ],
    "figure3_combined": [
        "fig3_combined_cindex.csv", "fig3_combined_os_cindex.csv",
        "fig3_combined_os_delta.csv",
    ],
    "within_cancer_km": [
        "fig2_km_stage_vs_risk_by_cancer.csv", "fig2_stage_vs_risk_cindex_by_cancer.csv",
    ],
    "published_scores": [
        "pubscore_cindex.csv", "pubscore_delta.csv", "pubscore_cox.csv",
        "pubscore_km.csv", "pubscore_cohort.csv",
    ],
    "figure4": [
        "fig4_trajectories_heatmap.csv", "fig4_km_data.csv", "fig4_cluster_severity.csv",
        "fig4_group_trajectories.csv", "fig4_slope_by_stage.csv", "fig4_silhouette.csv",
    ],
    "figure5": [
        "fig5_ps_predictions.csv", "fig5_robust_hits.csv", "fig5_km_top_hit.csv",
        "fig5_km_examples.csv", "fig5_top_hit_meta.csv", "fig5_love_smd.csv",
        "fig5_forest_headline.csv",
    ],
}

# Modules that checkpoint completed phases and resume after an interruption
# (within_cancer: full cohort, then modality cohort). FORCE/REGENERATE_ALL pass --restart.
RESUMABLE = {"within_cancer"}

# Extra CLI args per module, e.g. {"within_cancer": ["--min-patients", "30", "--min-events", "10"]}.
EXTRA_ARGS: dict[str, list[str]] = {}

# --- Run controls ---
REGENERATE_ALL = False        # True -> ignore existing outputs entirely
FORCE: set[str] = set()       # e.g. {"figure4"} to rebuild just that one
ONLY: set[str] = set()        # non-empty -> restrict the run to these modules

print(f"regenerate all: {REGENERATE_ALL}")
print(f"force:          {', '.join(sorted(FORCE)) or 'none'}")
print(f"only:           {', '.join(sorted(ONLY)) or 'all modules'}")

## Plan

What each module would do, before anything runs. Modules with every output present are skipped;
partial ones re-run in full and rewrite all of their CSVs. The exception is `within_cancer`, which
writes the full-cohort outputs as soon as that phase finishes and, after an interruption, resumes at
the modality-cohort phase (unless forced).

In [ ]:
def missing_outputs(module: str) -> list[str]:
    return [name for name in OUTPUTS[module]
            if not os.path.exists(os.path.join(FIGURE_DATA_DIR, name))]


modules = [m for m in OUTPUTS if not ONLY or m in ONLY]
plan = []

for module in modules:
    absent = missing_outputs(module)
    if REGENERATE_ALL:
        reason = "forced (REGENERATE_ALL)"
    elif module in FORCE:
        reason = "forced"
    elif not absent:
        reason = "skip"
    elif len(absent) == len(OUTPUTS[module]):
        reason = f"run — no outputs present ({len(absent)})"
    else:
        reason = f"run — {len(absent)}/{len(OUTPUTS[module])} outputs missing"
    plan.append((module, reason != "skip", absent, reason))

print(f"{'module':<16} {'action':<8} detail")
for module, will_run, absent, reason in plan:
    print(f"{module:<16} {'RUN' if will_run else 'skip':<8} {reason}")
    if will_run and absent and len(absent) < len(OUTPUTS[module]):
        print(f"{'':<25} missing: {', '.join(absent)}")

n_run = sum(1 for _, will_run, _, _ in plan if will_run)
print(f"\n{n_run} module(s) to run, {len(plan) - n_run} skipped")
if not n_run:
    print("Nothing to do — all figure data present. Set REGENERATE_ALL or FORCE to rebuild.")

## Run

Each module is `python -m figures.prep.<module>` with `cwd` set to the repo root. Output streams straight
through. A failure does **not** stop the queue — the modules are independent (they share input data,
not each other's outputs), so a broken figure5 says nothing about figure2. Everything is reported at
the end.

In [ ]:
results = []

for module, will_run, _absent, reason in plan:
    if not will_run:
        print(f"\n=== {module}: skipped ({reason}) ===")
        results.append((module, "skipped", 0.0))
        continue

    print(f"\n{'=' * 78}\n=== figures.prep.{module}  [{reason}]\n{'=' * 78}", flush=True)
    started = time.perf_counter()
    # Pipe the module's output into this cell: an uncaptured child writes to the
    # Jupyter server's terminal, so the notebook would show nothing until it exits.
    args = list(EXTRA_ARGS.get(module, []))
    # Resumable modules pick up completed phases of an interrupted run; a forced
    # rebuild must not reuse them.
    if module in RESUMABLE and reason.startswith("forced"):
        args.append("--restart")
    proc = subprocess.Popen(
        [sys.executable, "-m", f"figures.prep.{module}", *args],
        cwd=str(REPO_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    # Raw chunks rather than lines, so tqdm's carriage-return updates show live.
    while chunk := proc.stdout.read1(4096):
        sys.stdout.write(chunk.decode(errors="replace"))
        sys.stdout.flush()
    proc.wait()
    elapsed = time.perf_counter() - started
    status = "ok" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
    print(f"\n[{module}] {status} in {elapsed / 60:.1f} min")
    results.append((module, status, elapsed))

print(f"\n{'=' * 78}\n=== Run summary ===")
for module, status, elapsed in results:
    print(f"  {module:<16} {status}" + (f"  ({elapsed / 60:.1f} min)" if elapsed else ""))

## Verify

The end state on disk. A module that reported `ok` but still has missing outputs wrote fewer CSVs
than `OUTPUTS` claims — either the module changed or the list here is stale.

Zero-row CSVs are called out separately: `save_figure_data` writes them deliberately on
data-not-available paths and only logs a warning, and the R tier turns one into an empty tibble and
renders a placeholder panel without complaint. On disk they are indistinguishable from a real
result, so they are worth seeing here.

In [ ]:
import polars as pl

all_missing, empty = [], []

for module in modules:
    absent = missing_outputs(module)
    present = [n for n in OUTPUTS[module] if n not in absent]
    all_missing.extend(absent)

    for name in present:
        try:
            if pl.read_csv(os.path.join(FIGURE_DATA_DIR, name)).height == 0:
                empty.append(name)
        except Exception as exc:
            empty.append(f"{name} (unreadable: {type(exc).__name__})")

    flag = "ok " if not absent else "INCOMPLETE"
    print(f"[{flag:<10}] {module:<16} {len(present)}/{len(OUTPUTS[module])} outputs present")
    for name in absent:
        print(f"{'':<13} missing: {name}")

print(f"\n{sum(len(v) for v in OUTPUTS.values()) - len(all_missing)}"
      f"/{sum(len(v) for v in OUTPUTS.values())} figure-data CSVs present")

if empty:
    print(f"\n{len(empty)} file(s) with 0 rows — these render as placeholder panels:")
    for name in empty:
        print(f"  {name}")

failed = [m for m, status, _ in results if status.startswith("FAILED")]
if failed:
    print(f"\n{len(failed)} module(s) failed: {', '.join(failed)}")
elif all_missing:
    print("\nSome outputs are still missing — see above before running 4_figures/03.")
else:
    print("\nAll figure data present. Now run 03_render_figures.R to plot.")